# 02 Silver transform

Turns bronze text into clean, typed, trusted tables.

* **Master data** (products, stores, targets) is standardised and replaced on every run.
* **Exchange rates** become one rate per currency per day. Weekends and holidays use the last
  published rate, and each rate says whether it was published or carried forward.
* **Sales lines** from the tills and the webshop are combined into one table. Only bronze rows
  newer than the last run are processed (a watermark in `meta.watermarks`). Each line is
  checked against the rules in `nb_lib_retail`; lines that break a rule go to
  `silver.sales_quarantine` with the reason, never silently dropped.
* Lines are written with a Delta **MERGE** on `sales_line_id`, and the newest export of a line
  wins. A re-sent file changes nothing, and a webshop order that goes from pending to
  completed or cancelled is updated in place. Running the notebook twice gives the same result.

In [ ]:
run_date = ""                # yyyy-mm-dd, defaults to today
batch_id = ""

In [ ]:
import datetime as dt

from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [ ]:
%run nb_lib_retail

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
run_date = run_date or dt.date.today().isoformat()

## Master data and exchange rates

In [ ]:
products = standardize_products(spark.table("bronze.products_raw"))
stores = standardize_stores(spark.table("bronze.stores_raw"))
targets = standardize_targets(spark.table("bronze.sales_targets_raw"))

fx_raw = spark.table("bronze.fx_rates_raw")
fx_start = fx_raw.agg(F.min(F.to_date("rate_date"))).first()[0]
fx = fx_daily(fx_raw, fx_start, dt.date.fromisoformat(run_date))

for name, df in [("products", products), ("stores", stores), ("sales_targets", targets), ("fx_rates_daily", fx)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{name}")

products, stores, fx = spark.table("silver.products"), spark.table("silver.stores"), spark.table("silver.fx_rates_daily")
print("carried forward rates:", fx.where("rate_source = 'carried forward'").count())

## New sales lines since the last run

In [ ]:
spark.sql("CREATE TABLE IF NOT EXISTS meta.watermarks (name STRING, value TIMESTAMP, updated_at TIMESTAMP) USING DELTA")
spark.sql("""CREATE TABLE IF NOT EXISTS meta.run_log (
    batch_id STRING, notebook STRING, metric STRING, value BIGINT, logged_at TIMESTAMP) USING DELTA""")

watermark = spark.sql("SELECT max(value) FROM meta.watermarks WHERE name = 'silver_sales'").first()[0]
pos_new = spark.table("bronze.pos_sales_raw")
web_new = spark.table("bronze.web_orders_raw")
if watermark is not None:
    pos_new = pos_new.where(F.col("_ingested_at") > F.lit(watermark))
    web_new = web_new.where(F.col("_ingested_at") > F.lit(watermark))
new_max = pos_new.select("_ingested_at").unionByName(web_new.select("_ingested_at")).agg(F.max("_ingested_at")).first()[0]
print(f"watermark {watermark}, newest bronze row {new_max}")

In [ ]:
if new_max is None:
    print("nothing new in bronze")
else:
    valid, quarantine, incoming, deduped = silver_sales_from_bronze(pos_new, web_new, stores, products, fx, run_date)
    valid, quarantine = valid.cache(), quarantine.cache()
    stats = {
        "incoming_lines": incoming.count(),
        "after_dedup": deduped.count(),
        "valid": valid.count(),
        "quarantined": quarantine.count(),
        "store_codes_fixed": incoming.where("_fixed_store_code").count(),
    }
    stats["duplicates_removed"] = stats["incoming_lines"] - stats["after_dedup"]
    print(stats)
    display(quarantine.groupBy("dq_reason").count().orderBy(F.desc("count")))

## MERGE into silver

Valid lines are upserted into `silver.sales_lines`; quarantined lines into
`silver.sales_quarantine`. If a newer version of a line moves it from one table to the other
(for example a corrected re-send), it is removed from the table it was in, so a line is
always in exactly one place.

In [ ]:
def upsert(df, table, key="sales_line_id"):
    if not spark.catalog.tableExists(table):
        df.write.format("delta").saveAsTable(table)
        return
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), f"t.{key} = s.{key}")
        .whenMatchedUpdateAll(condition="s._version_ts >= t._version_ts")
        .whenNotMatchedInsertAll()
        .execute())


def remove_moved(table, moved, key="sales_line_id"):
    if spark.catalog.tableExists(table):
        (DeltaTable.forName(spark, table).alias("t")
            .merge(moved.select(key, "_version_ts").alias("s"), f"t.{key} = s.{key}")
            .whenMatchedDelete(condition="s._version_ts >= t._version_ts")
            .execute())


if new_max is not None:
    upsert(valid, "silver.sales_lines")
    remove_moved("silver.sales_quarantine", valid)
    upsert(quarantine, "silver.sales_quarantine")
    remove_moved("silver.sales_lines", quarantine)

    spark.createDataFrame([("silver_sales", new_max, dt.datetime.utcnow())], "name string, value timestamp, updated_at timestamp") \
         .write.format("delta").mode("append").saveAsTable("meta.watermarks")
    now = dt.datetime.utcnow()
    spark.createDataFrame([(batch_id, "02_silver_transform", k, int(v), now) for k, v in stats.items()],
                          "batch_id string, notebook string, metric string, value bigint, logged_at timestamp") \
         .write.format("delta").mode("append").saveAsTable("meta.run_log")

In [ ]:
display(spark.sql("""
    SELECT source, order_status, count(*) AS lines, round(sum(net_amount_dkk), 2) AS net_amount_dkk
    FROM silver.sales_lines GROUP BY source, order_status ORDER BY source, order_status
"""))